In [29]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

# Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# dx = 1 km; Np = 1M; Nt = 5 min
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_5min.nc') #***
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_5min_1e6.nc') #***
res='1km';t_res='5min'
Np_str='1e6'


# # dx = 1km; Np = 50M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
# res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 100M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_100M.nc') #***
# res='1km'; t_res='1min'; Np_str='100e6'

# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# # # parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# # # parcel=parcel.isel(time=np.arange(0,400+1))
# res='250m'

In [30]:
times=data['time'].values/(1e9 * 60); times=times.astype(float);
minutes=1/times[1] #1 / minutes per timestep = timesteps per minute

In [31]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(PlottingFunctions, inspect.isfunction)]
# functions

In [32]:
#JOB ARRAY SETUP
job_array=False;index_adjust=0
# job_array=True
##############################*#*

if job_array==True:
    num_jobs=60 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
    total_elements=len(parcel['xh']) #total num of variables

    if num_jobs >= total_elements:
        raise ValueError("Number of jobs cannot be greater than or equal to total elements.")
    
    job_range = total_elements // num_jobs  # Base size for each chunk
    remaining = total_elements % num_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id, num_jobs):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_jobs - 1: 
            end_job = total_elements #- 1
        return start_job, end_job
    # def job_testing():
    #     #TESTING
    #     start=[];end=[]
    #     for job_id in range(1,num_jobs+1):
    #         start_job, end_job = get_job_range(job_id)
    #         print(start_job,end_job)
    #         start.append(start_job)
    #         end.append(end_job)
    #     print(np.all(start!=end))
    #     print(len(np.unique(start))==len(start))
    #     print(len(np.unique(end))==len(end))
    # job_testing()
    
    job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    if job_id==0: job_id=1
    start_job, end_job = get_job_range(job_id, num_jobs)
    index_adjust=start_job
    print(f'start_job = {start_job}, end_job = {end_job}')

In [34]:
if job_array==True:
    #Indexing Array with JobArray
    parcel=parcel.isel(xh=slice(start_job,end_job))
    #(for 150_000_000 parcels use 500-1000 jobs)

In [35]:
# Reading Back Data Later
##############
def make_data_dict(var_names,read_type):
    if read_type=='h5py':
        with h5py.File(in_file, 'r') as f:
            if job_array==True:
                data_dict = {var_name: f[var_name][:,start_job:end_job] for var_name in var_names}
            elif job_array==False:
                data_dict = {var_name: f[var_name][:] for var_name in var_names}
            
    elif read_type=='xarray':
        in_data = xr.open_dataset(
            in_file,
            engine='h5netcdf',
            phony_dims='sort',
            chunks={'phony_dim_0': 100, 'phony_dim_1': 1_000_000} 
        )
        if job_array==True:
            data_dict = {k: in_data[k][:,start_job:end_job].compute().data for k in var_names}
        elif job_array==False:
            data_dict = {k: in_data[k][:].compute().data for k in var_names}
    return data_dict

# read_type='xarray'
read_type='h5py'

In [36]:
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'lagrangian_binary_array_{res}_{t_res}_{Np_str}.h5'

var_names = ['A_g', 'A_c', 'W', 'QCQI', 'Z', 'Y', 'X', 'z']
data_dict = make_data_dict(var_names,read_type)
A_g, A_c, W, QCQI, Z, Y, X, parcel_z = (data_dict[k] for k in var_names)

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)
check_memory(globals())

Top 10 objects with highest memory usage
{'A_g': '1064.0 MB', 'A_c': '1064.0 MB', 'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'W': '532.0 MB', 'QCQI': '532.0 MB', 'parcel_z': '532.0 MB', 'VMF_G': '532.0 MB', 'VMF_C': '532.0 MB'}

7.98 GB in use overall


In [37]:
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'VMF_binary_array_{res}_{t_res}_{Np_str}.h5'

var_names = ['VMF_G','VMF_C']
data_dict = make_data_dict(var_names,read_type)
VMF_G, VMF_C = (data_dict[k] for k in var_names)

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)
check_memory(globals())

Top 10 objects with highest memory usage
{'A_g': '1064.0 MB', 'A_c': '1064.0 MB', 'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'W': '532.0 MB', 'QCQI': '532.0 MB', 'parcel_z': '532.0 MB', 'VMF_G': '532.0 MB', 'VMF_C': '532.0 MB'}

7.98 GB in use overall


In [38]:
#TRACKED TRAJECTORY ENTRAINMENT/DETRAINMENT
################################################################

In [39]:
########################
#READING BACK IN
def LoadFinalData(in_file):
    dict = {}
    with h5py.File(in_file, 'r') as f:
        for key in f.keys():
            dict[key] = f[key][:]
    return dict

def LoadAllCloudBase():
    dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/'
    in_file = dir2 + f"all_cloudbase_{res}_{t_res}_{Np_str}.pkl"
    with open(in_file, 'rb') as f:
        all_cloudbase = pickle.load(f)
    return(all_cloudbase)
min_all_cloudbase=np.nanmin(LoadAllCloudBase())
print(f"Minimum Cloudbase is: {min_all_cloudbase}\n")

dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/'
in_file=dir2+f"parcel_tracking_SUBSET_{res}_{t_res}_{Np_str}"
final_dict=LoadFinalData(in_file)


#DYNAMICALLY CREATING VARIABLES
for key, value in final_dict.items():
    globals()[key] = value

# #DYNAMICALLY PRINTING VARIABLE SIZES
# for key in final_dict:
#     print(f"{key} has {final_dict[key].shape[0]} parcels")

# PRINTING VARIABLE SIZES (ONE BY ONE)
print(f'ALL: {len(CL_ALL_out_arr)} CL parcels and {len(nonCL_ALL_out_arr)} nonCL parcels')
print(f'SHALLOW: {len(CL_SHALLOW_out_arr)} CL parcels and {len(nonCL_SHALLOW_out_arr)} nonCL parcels')
print(f'DEEP: {len(CL_DEEP_out_arr)} CL parcels and {len(nonCL_DEEP_out_arr)} nonCL parcels')
print('\n')
print(f'ALL: {len(SBZ_ALL_out_arr)} SBZ parcels and {len(nonSBZ_ALL_out_arr)} nonSBZ parcels')
print(f'SHALLOW: {len(SBZ_SHALLOW_out_arr)} SBZ parcels and {len(nonSBZ_SHALLOW_out_arr)} nonSBZ parcels')
print(f'DEEP: {len(SBZ_DEEP_out_arr)} SBZ parcels and {len(nonSBZ_DEEP_out_arr)} nonSBZ parcels')
print('\n')
print(f'ALL: {len(ColdPool_ALL_out_arr)} ColdPool parcels')
print(f'SHALLOW: {len(ColdPool_SHALLOW_out_arr)} ColdPool parcels')
print(f'DEEP: {len(ColdPool_DEEP_out_arr)} ColdPool parcels')


#APPLYING JOB ARRAY
if job_array==True:
    print('APPLYING JOB ARRAY')
    def job_filter(arr):
        return arr[(arr[:,0]>=start_job)&(arr[:,0]<end_job)]
    for name in [
        'CL_ALL_out_arr', 'CL_ALL_out_arr',
        'CL_SHALLOW_out_arr', 'CL_SHALLOW_out_arr',
        'CL_DEEP_out_arr', 'nonCL_DEEP_out_arr',
        'SBZ_ALL_out_arr', 'nonSBZ_ALL_out_arr',
        'SBZ_SHALLOW_out_arr', 'nonSBZ_SHALLOW_out_arr',
        'SBZ_DEEP_out_arr', 'nonSBZ_DEEP_out_arr',
        'ColdPool_ALL_out_arr', 'ColdPool_SHALLOW_out_arr', 'ColdPool_DEEP_out_arr'
    ]:
        globals()[name] = job_filter(globals()[name])

Minimum Cloudbase is: 1.2463867664337158

ALL: 14778 CL parcels and 11105 nonCL parcels
SHALLOW: 10167 CL parcels and 8454 nonCL parcels
DEEP: 1501 CL parcels and 1119 nonCL parcels


ALL: 818 SBZ parcels and 13960 nonSBZ parcels
SHALLOW: 426 SBZ parcels and 9741 nonSBZ parcels
DEEP: 188 SBZ parcels and 1313 nonSBZ parcels


ALL: 13960 ColdPool parcels
SHALLOW: 9741 ColdPool parcels
DEEP: 1313 ColdPool parcels


In [40]:
##########################################################################################
#(ALL, SHALLOW, DEEP) CL vs NonCL Tracked VMF Profiles

In [41]:
#MAKING ENTRAINMENT PROFILE ARRAY FUNCTION
    
def TrackedVMFProfile(type1,type2,type3):

    if type2=='general':
        profile_array_VMF=VMF_G.copy() 
    elif type2=='cloudy':
        profile_array_VMF=VMF_C.copy() 

    out_arr=globals()[f"{type3}_{type1.upper()}_out_arr"]
     
    zhs=data['zh'].values
    VMF_profile=np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    VMF_profile[:,2]=zhs

    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust] #JOBARRAY
        ys=Y[ts,p-index_adjust] #JOBARRAY
        xs=X[ts,p-index_adjust] #JOBARRAY
        
        vars_VMF=profile_array_VMF[ts,p-index_adjust] #JOBARRAY
        np.add.at(VMF_profile[:, 0], zs, vars_VMF)
        np.add.at(VMF_profile[:, 1], zs, 1)

    
    print('done\n')
    return VMF_profile

In [46]:
for type2 in ["general","cloudy"]:
    print(f"{type2}\n")
    CL_ALL_VMF = TrackedVMFProfile(type1='all',type2=type2,type3='CL')
    CL_SHALLOW_VMF = TrackedVMFProfile(type1='shallow',type2=type2,type3='CL')
    CL_DEEP_VMF = TrackedVMFProfile(type1='deep',type2=type2,type3='CL')
    
    #SAVING
    import h5py
    filePath=dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
    filePath+=f"CL_tracked_profiles_VMF_{res}_{t_res}_{Np_str}"
    if job_array==True:
        filePath+=f"_{job_id}.h5"
    elif job_array==False:
        filePath+=f".h5"
    with h5py.File(filePath, 'w') as h5f:
        h5f.create_dataset('CL_ALL_VMF', data=CL_ALL_VMF)
        h5f.create_dataset('CL_SHALLOW_VMF', data=CL_SHALLOW_VMF)
        h5f.create_dataset('CL_DEEP_VMF', data=CL_DEEP_VMF)

general

done

done

done

cloudy

done

done

done



In [47]:
for type2 in ["general","cloudy"]:
    print(f"{type2}\n")
    nonCL_ALL_VMF = TrackedVMFProfile(type1='all',type2=type2,type3='nonCL')
    nonCL_SHALLOW_VMF = TrackedVMFProfile(type1='shallow',type2=type2,type3='nonCL')
    nonCL_DEEP_VMF = TrackedVMFProfile(type1='deep',type2=type2,type3='nonCL')
    
    #SAVING
    import h5py
    filePath=dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
    filePath+=f"nonCL_tracked_profiles_VMF_{res}_{t_res}_{Np_str}"
    if job_array==True:
        filePath+=f"_{job_id}.h5"
    elif job_array==False:
        filePath+=f".h5"
    with h5py.File(filePath, 'w') as h5f:
        h5f.create_dataset('nonCL_ALL_VMF', data=nonCL_ALL_VMF)
        h5f.create_dataset('nonCL_SHALLOW_VMF', data=nonCL_SHALLOW_VMF)
        h5f.create_dataset('nonCL_DEEP_VMF', data=nonCL_DEEP_VMF)

general

done

done

done

cloudy

done

done

done



In [48]:
##########################################################################################
#SBZ vs nonSBZ Tracked Entrainment Profiles
for type2 in ["general","cloudy"]:
    print(f"{type2}\n")
    SBZ_ALL_VMF = TrackedVMFProfile(type1='all',type2=type2,type3='SBZ')
    SBZ_SHALLOW_VMF = TrackedVMFProfile(type1='shallow',type2=type2,type3='SBZ')
    SBZ_DEEP_VMF = TrackedVMFProfile(type1='deep',type2=type2,type3='SBZ')
    
    # Define the file path
    filePath=dir+f"Project_Algorithms/Entrainment/trackout/{type2}_"
    filePath+=f"SBZ_tracked_profiles_VMF_{res}_{t_res}_{Np_str}"
    if job_array==True:
        filePath+=f"_{job_id}.h5"
    elif job_array==False:
        filePath+=f".h5"
    # Write datasets to the HDF5 file
    with h5py.File(filePath, 'w') as h5f:
        h5f.create_dataset('SBZ_ALL_VMF', data=SBZ_ALL_VMF)
        h5f.create_dataset('SBZ_SHALLOW_VMF', data=SBZ_SHALLOW_VMF)
        h5f.create_dataset('SBZ_DEEP_VMF', data=SBZ_DEEP_VMF)


general

done

done

done

cloudy

done

done

done



In [49]:
for type2 in ["general","cloudy"]:
    print(f"{type2}\n")
    nonSBZ_ALL_VMF = TrackedVMFProfile(type1='all',type2=type2,type3='SBZ')
    nonSBZ_SHALLOW_VMF = TrackedVMFProfile(type1='shallow',type2=type2,type3='SBZ')
    nonSBZ_DEEP_VMF = TrackedVMFProfile(type1='deep',type2=type2,type3='SBZ')
    
    # Define the file path
    filePath=dir+f"Project_Algorithms/Entrainment/trackout/{type2}_"
    filePath+=f"nonSBZ_tracked_profiles_VMF_{res}_{t_res}_{Np_str}"
    if job_array==True:
        filePath+=f"_{job_id}.h5"
    elif job_array==False:
        filePath+=f".h5"
    # Write datasets to the HDF5 file
    with h5py.File(filePath, 'w') as h5f:
        h5f.create_dataset('nonSBZ_ALL_VMF', data=nonSBZ_ALL_VMF)
        h5f.create_dataset('nonSBZ_SHALLOW_VMF', data=nonSBZ_SHALLOW_VMF)
        h5f.create_dataset('nonSBZ_DEEP_VMF', data=nonSBZ_DEEP_VMF)

general

done

done

done

cloudy

done

done

done



In [51]:
#ColdPool
################################################################

for type2 in ["general","cloudy"]:
    print(f"{type2}\n")
    ColdPool_ALL_VMF = TrackedVMFProfile(type1='all',type2=type2,type3='ColdPool')
    ColdPool_SHALLOW_VMF = TrackedVMFProfile(type1='shallow',type2=type2,type3='ColdPool')
    ColdPool_DEEP_VMF = TrackedVMFProfile(type1='deep',type2=type2,type3='ColdPool')
    
    # Define the file path
    filePath=dir+f"Project_Algorithms/Entrainment/trackout/{type2}_"
    filePath+=f"ColdPool_tracked_profiles_VMF_{res}_{t_res}_{Np_str}"
    if job_array==True:
        filePath+=f"_{job_id}.h5"
    elif job_array==False:
        filePath+=f".h5"
    # Write datasets to the HDF5 file
    with h5py.File(filePath, 'w') as h5f:
        h5f.create_dataset('ColdPool_ALL_VMF', data=ColdPool_ALL_VMF)
        h5f.create_dataset('ColdPool_SHALLOW_VMF', data=ColdPool_SHALLOW_VMF)
        h5f.create_dataset('ColdPool_DEEP_VMF', data=ColdPool_DEEP_VMF)

general

done

done

done

cloudy

done

done

done



In [9]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [ ]:
if recombine==True:
    def get_profiles(job_id):
        # Define file path
        input_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
        input_file+=f"CL_tracked_profiles_VMF_{res}_{t_res}_{Np_str}_{job_id}.h5"
        
        # Load CL-tracked profiles and store them in a dictionary
        data_dict = {}
        
        with h5py.File(input_file, 'r') as h5f:
            for key in h5f.keys():
                data_dict[key] = h5f[key][:]  # Load each dataset into the dictionary
        return data_dict
    
    job_id=1
    dict1=get_profiles(job_id)
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
        dict2=get_profiles(job_id)
        for var in dict2:
            dict1[var][:,0:1+1]+=dict2[var][:,0:1+1]
    
    
    #SAVING INTO FINAL FORM
    output_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'+f"CL_tracked_profiles_VMF_{type2}.h5"
    with h5py.File(output_file, 'w') as f:
        for var in dict1:
            profile_var = dict1[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

In [88]:
if recombine==True:
    def get_profiles(job_id):
        # Define file path
        input_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
        input_file+=f"nonCL_tracked_profiles_VMF_{res}_{t_res}_{Np_str}_{job_id}.h5"
        
        # Load CL-tracked profiles and store them in a dictionary
        data_dict = {}
        
        with h5py.File(input_file, 'r') as h5f:
            for key in h5f.keys():
                data_dict[key] = h5f[key][:]  # Load each dataset into the dictionary
        return data_dict
    
    job_id=1
    dict1=get_profiles(job_id)
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
        dict2=get_profiles(job_id)
        for var in dict2:
            dict1[var][:,0:1+1]+=dict2[var][:,0:1+1]
    
    
    #SAVING INTO FINAL FORM
    output_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'+f"nonCL_tracked_profiles_VMF_{type2}.h5"
    with h5py.File(output_file, 'w') as f:
        for var in dict1:
            profile_var = dict1[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

job_id = 10
job_id = 20
job_id = 30
job_id = 40
job_id = 50
job_id = 60


In [ ]:
if recombine==True:
    def get_profiles(job_id):
        # Define file path
        input_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
        input_file+=f'SBZ_tracked_profiles_VMF_{res}_{t_res}_{Np_str}_{job_id}.h5'
        
        # Load CL-tracked profiles and store them in a dictionary
        data_dict = {}
        
        with h5py.File(input_file, 'r') as h5f:
            for key in h5f.keys():
                data_dict[key] = h5f[key][:]  # Load each dataset into the dictionary
        return data_dict
    
    job_id=1
    dict1=get_profiles(job_id)
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
        dict2=get_profiles(job_id)
        for var in dict2:
            dict1[var][:,0:1+1]+=dict2[var][:,0:1+1]
    
    
    #SAVING INTO FINAL FORM
    output_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'+f'SBZ_tracked_profiles_VMF.h5'
    with h5py.File(output_file, 'w') as f:
        for var in dict1:
            profile_var = dict1[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

In [ ]:
if recombine==True:
    def get_profiles(job_id):
        # Define file path
        input_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
        input_file+=f'nonSBZ_tracked_profiles_VMF_{res}_{t_res}_{Np_str}_{job_id}.h5'
        
        # Load CL-tracked profiles and store them in a dictionary
        data_dict = {}
        
        with h5py.File(input_file, 'r') as h5f:
            for key in h5f.keys():
                data_dict[key] = h5f[key][:]  # Load each dataset into the dictionary
        return data_dict
    
    job_id=1
    dict1=get_profiles(job_id)
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
        dict2=get_profiles(job_id)
        for var in dict2:
            dict1[var][:,0:1+1]+=dict2[var][:,0:1+1]
    
    
    #SAVING INTO FINAL FORM
    output_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'+f'nonSBZ_tracked_profiles_VMF.h5'
    with h5py.File(output_file, 'w') as f:
        for var in dict1:
            profile_var = dict1[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

In [116]:
if recombine==True:
    def get_profiles(job_id):
        # Define file path
        input_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'
        input_file+=f'ColdPool_tracked_profiles_VMF_{res}_{t_res}_{Np_str}_{job_id}.h5'
        
        # Load CL-tracked profiles and store them in a dictionary
        data_dict = {}
        
        with h5py.File(input_file, 'r') as h5f:
            for key in h5f.keys():
                data_dict[key] = h5f[key][:]  # Load each dataset into the dictionary
        return data_dict
    
    job_id=1
    dict1=get_profiles(job_id)
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
        dict2=get_profiles(job_id)
        for var in dict2:
            dict1[var][:,0:1+1]+=dict2[var][:,0:1+1]
    
    
    #SAVING INTO FINAL FORM
    output_file = dir+f'Project_Algorithms/Entrainment/trackout/{type2}_'+f'ColdPool_tracked_profiles_VMF_{res}_{t_res}_{Np_str}.h5'
    with h5py.File(output_file, 'w') as f:
        for var in dict1:
            profile_var = dict1[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

job_id = 10
job_id = 20
job_id = 30
job_id = 40
job_id = 50
job_id = 60
